Deze code kan je doorlopen om van de nc bestanden van biooracle een dataset te maken met de juiste locaties zoals onze oorspronkelijke dataset

In [1]:
import pandas as pd
import xarray as xr
import glob
import numpy as np #

In [2]:
df_biooracle = pd.read_csv("Data/Diversity_data_with_biooracle_2010.csv")
df_biooracle.columns

Index(['marine_species_richness', 'PD', 'co1_genetic_diversity_mean',
       'long_deg', 'lat_deg', 'chl_max', 'chl_mean', 'chl_min', 'clt_max',
       'clt_mean', 'clt_min', 'currentdirection_max', 'currentdirection_mean',
       'currentdirection_min', 'currentvelocity_ltmax', 'currentvelocity_max',
       'currentvelocity_mean', 'currentvelocity_min', 'dfe_max', 'dfe_mean',
       'dfe_min', 'kdpar_max', 'kdpar_mean', 'kdpar_min', 'mlotst_max',
       'mlotst_mean', 'mlotst_min', 'no3_max', 'no3_mean', 'no3_min', 'o2_max',
       'o2_mean', 'o2_min', 'par_mean', 'par_min', 'phyc_max', 'phyc_mean',
       'phyc_min', 'ph_max', 'ph_mean', 'ph_min', 'po4_max', 'po4_mean',
       'po4_min', 'salinity_ltmax', 'salinity_ltmin', 'salinity_max',
       'salinity_mean', 'salinity_min', 'salinity_range', 'siconc_max',
       'siconc_mean', 'siconc_min', 'sithick_max', 'sithick_mean',
       'sithick_min', 'si_max', 'si_mean', 'si_min', 'tas_max', 'tas_mean',
       'tas_min', 'terrain_charact

maak een dataset aan met enkel de locaties (dit hebben we straks nodig)

In [3]:
df_locations = df_biooracle[["long_deg", "lat_deg"]]
df_locations.head(10)

,long_deg,lat_deg
0,-171.74579,-71.23536
1,-167.88679,-71.23536
2,-164.02779,-71.23536
3,-160.16879,-71.23536
4,-156.30979,-71.23536
5,-152.45079,-71.23536
6,-148.59179,-71.23536
7,-144.73279,-71.23536
8,-140.87379,-71.23536
9,-59.83479,-71.23536


In [4]:
df_env = pd.read_csv("Data/Diversity_data_with_env.csv")
df_env.head()

,grid_id,long,lat,marine_species_richness,PD,co1_genetic_diversity_mean,long_deg,lat_deg,long_dd,lat_dd,...,Shelf,Slope,Abyssal,TidalRange,Coral,Estuary,Seamount,MPA,matched_CenterLong,matched_CenterLat
0,1,-17174579,-7123536,44,11.830844,0.007171,-171.74579,-71.23536,-171.74579,-71.23536,...,0.0,730.228795,263.361205,-9999.0,0.0,0.0,0,NaN,-171.75,-71.25
1,2,-16788679,-7123536,43,11.830844,0.007171,-167.88679,-71.23536,-167.88679,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-167.75,-71.25
2,3,-16402779,-7123536,44,12.650381,0.007499,-164.02779,-71.23536,-164.02779,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-164.25,-71.25
3,4,-16016879,-7123536,44,12.650381,0.007499,-160.16879,-71.23536,-160.16879,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-160.25,-71.25
4,5,-15630979,-7123536,45,12.650381,0.007499,-156.30979,-71.23536,-156.30979,-71.23536,...,0.0,0.000000,993.590000,-9999.0,0.0,0.0,0,NaN,-156.25,-71.25


Verzamel al je nc bestanden in een map zoals de map Data.
Ik zou per biodiversiteitsmaat een andere naam aan de map geven (mss ncPD, ncGD en ncSR?)

In [5]:
files = glob.glob("ncGD_SSP126/*.nc")

Van alle aparte nc bestandjes 1 geheel maken 

In [6]:
datasets = [xr.open_dataset(f) for f in files]
ds = xr.merge(datasets)

print(ds)

<xarray.Dataset> Size: 3GB
Dimensions:       (time: 1, latitude: 3600, longitude: 7200)
Coordinates:
  * time          (time) datetime64[ns] 8B 2050-01-01
  * latitude      (latitude) float32 14kB -89.97 -89.93 -89.88 ... 89.93 89.97
  * longitude     (longitude) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables: (12/14)
    chl_mean      (time, latitude, longitude) float64 207MB ...
    clt_mean      (time, latitude, longitude) float64 207MB ...
    dfe_mean      (time, latitude, longitude) float64 207MB ...
    mlotst_mean   (time, latitude, longitude) float64 207MB ...
    no3_mean      (time, latitude, longitude) float64 207MB ...
    o2_mean       (time, latitude, longitude) float64 207MB ...
    ...            ...
    po4_mean      (time, latitude, longitude) float64 207MB ...
    siconc_mean   (time, latitude, longitude) float64 207MB ...
    sithick_mean  (time, latitude, longitude) float64 207MB ...
    so_mean       (time, latitude, longitude) float64 207MB ...

De dataset omvormen tot een dataframe waarmee je kan verder werken om aanpassingen aan te brengen 

In [7]:
df = ds.to_dataframe().reset_index()

In [8]:
len(df)  # is nog veel te lang dus veel locaties moeten nog wegvallen => komt straks

25920000

In [9]:
df.head()    # je ziet dat longitude en latitude de variabelen zijn maar we hebben long_deg en lat_deg nodig

,time,latitude,longitude,chl_mean,clt_mean,dfe_mean,mlotst_mean,no3_mean,o2_mean,phyc_mean,ph_mean,po4_mean,siconc_mean,sithick_mean,so_mean,tas_mean,thetao_mean
0,2050-01-01,-89.974998,-179.975006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2050-01-01,-89.974998,-179.925003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2050-01-01,-89.974998,-179.875000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2050-01-01,-89.974998,-179.824997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2050-01-01,-89.974998,-179.774994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df = df.rename(columns={"longitude": "long_deg", "latitude": "lat_deg"})

Je moet long_deg en lat_deg afronden tot op 5 cijfers na de komma anders zit je met andere komma getallen dan bij de oorspronkelijke dataset

In [11]:
print(df[["lat_deg", "long_deg"]].dtypes)
df["lat_deg"] = df["lat_deg"].astype("float64").round(5)
df["long_deg"] = df["long_deg"].astype("float64").round(5)
print(df[["lat_deg", "long_deg"]].head())

lat_deg     float32
long_deg    float32
dtype: object
   lat_deg   long_deg
0  -89.975 -179.97501
1  -89.975 -179.92500
2  -89.975 -179.87500
3  -89.975 -179.82500
4  -89.975 -179.77499


De volgende code is om de locaties te laten kloppen zoals in de oorspronkelijke dataset

In [12]:
# functie om dichtstbijzijnde waarde te vinden
def nearest(array, values):
    """
    array: 1D array van rasterwaarden (lat of long)
    values: array van gewenste waarden
    return: voor elke waarde in values de dichtstbijzijnde in array
    """
    array = np.array(array)
    values = np.array(values)
    idx = np.abs(array[:, None] - values).argmin(axis=0)
    return array[idx]

# vind dichtstbijzijnde lat/lon
df_locations["lat_match"] = nearest(df["lat_deg"].unique(), df_locations["lat_deg"])
df_locations["long_match"] = nearest(df["long_deg"].unique(), df_locations["long_deg"])

# merge op de “nearest” kolommen
df_subset = df.merge(
    df_locations,
    left_on=["lat_deg", "long_deg"],
    right_on=["lat_match", "long_match"],
    how="inner"
)

C:\Users\norah\AppData\Local\Temp\ipykernel_10200\4132841367.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_locations["lat_match"] = nearest(df["lat_deg"].unique(), df_locations["lat_deg"])
C:\Users\norah\AppData\Local\Temp\ipykernel_10200\4132841367.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_locations["long_match"] = nearest(df["long_deg"].unique(), df_locations["long_deg"])


In [13]:
df_subset.head()  #je ziet dat long_deg_y en lat_deg_y exact overeenkomen met de originele dataset => al de rest
# van long en lat variabelen kan je verwijderen 

,time,lat_deg_x,long_deg_x,chl_mean,clt_mean,dfe_mean,mlotst_mean,no3_mean,o2_mean,phyc_mean,...,po4_mean,siconc_mean,sithick_mean,so_mean,tas_mean,thetao_mean,long_deg_y,lat_deg_y,lat_match,long_match
0,2050-01-01,-71.225,-171.72501,0.264738,0.685382,0.000333,59.873447,30.339450,317.311825,1.462501,...,2.127915,0.727800,0.707284,33.942649,-11.655294,-1.580884,-171.74579,-71.23536,-71.225,-171.72501
1,2050-01-01,-71.225,-167.87500,0.279545,0.698337,0.000307,59.631997,30.320359,319.230605,1.453052,...,2.126779,0.720161,0.725998,33.953235,-11.305795,-1.583080,-167.88679,-71.23536,-71.225,-167.87500
2,2050-01-01,-71.225,-164.02499,0.272520,0.703587,0.000304,52.292932,30.218967,318.271827,1.395905,...,2.119736,0.681336,0.676522,33.936083,-10.861727,-1.538709,-164.02779,-71.23536,-71.225,-164.02499
3,2050-01-01,-71.225,-160.17500,0.297756,0.698612,0.000315,31.966103,29.987898,320.985091,1.454854,...,2.105292,0.677055,0.682275,33.892117,-10.390290,-1.531541,-160.16879,-71.23536,-71.225,-160.17500
4,2050-01-01,-71.225,-156.32500,0.304306,0.693806,0.000356,38.777512,29.912720,319.709665,1.456157,...,2.100458,0.662759,0.682666,33.845108,-9.906712,-1.513181,-156.30979,-71.23536,-71.225,-156.32500


In [14]:
df_subset = df_subset.drop(["lat_match","long_match", "long_deg_x", "lat_deg_x", "time"], axis = 1)
df_subset.head()

,chl_mean,clt_mean,dfe_mean,mlotst_mean,no3_mean,o2_mean,phyc_mean,ph_mean,po4_mean,siconc_mean,sithick_mean,so_mean,tas_mean,thetao_mean,long_deg_y,lat_deg_y
0,0.264738,0.685382,0.000333,59.873447,30.339450,317.311825,1.462501,7.956235,2.127915,0.727800,0.707284,33.942649,-11.655294,-1.580884,-171.74579,-71.23536
1,0.279545,0.698337,0.000307,59.631997,30.320359,319.230605,1.453052,7.957836,2.126779,0.720161,0.725998,33.953235,-11.305795,-1.583080,-167.88679,-71.23536
2,0.272520,0.703587,0.000304,52.292932,30.218967,318.271827,1.395905,7.957967,2.119736,0.681336,0.676522,33.936083,-10.861727,-1.538709,-164.02779,-71.23536
3,0.297756,0.698612,0.000315,31.966103,29.987898,320.985091,1.454854,7.960676,2.105292,0.677055,0.682275,33.892117,-10.390290,-1.531541,-160.16879,-71.23536
4,0.304306,0.693806,0.000356,38.777512,29.912720,319.709665,1.456157,7.960031,2.100458,0.662759,0.682666,33.845108,-9.906712,-1.513181,-156.30979,-71.23536


In [15]:
len(df_subset)  # lengte komt exact overeen met oorspronkelijke dataset

2452

In [16]:
df_subset = df_subset.rename(columns={"long_deg_y": "long_deg", "lat_deg_y": "lat_deg"})
df_subset.head()

,chl_mean,clt_mean,dfe_mean,mlotst_mean,no3_mean,o2_mean,phyc_mean,ph_mean,po4_mean,siconc_mean,sithick_mean,so_mean,tas_mean,thetao_mean,long_deg,lat_deg
0,0.264738,0.685382,0.000333,59.873447,30.339450,317.311825,1.462501,7.956235,2.127915,0.727800,0.707284,33.942649,-11.655294,-1.580884,-171.74579,-71.23536
1,0.279545,0.698337,0.000307,59.631997,30.320359,319.230605,1.453052,7.957836,2.126779,0.720161,0.725998,33.953235,-11.305795,-1.583080,-167.88679,-71.23536
2,0.272520,0.703587,0.000304,52.292932,30.218967,318.271827,1.395905,7.957967,2.119736,0.681336,0.676522,33.936083,-10.861727,-1.538709,-164.02779,-71.23536
3,0.297756,0.698612,0.000315,31.966103,29.987898,320.985091,1.454854,7.960676,2.105292,0.677055,0.682275,33.892117,-10.390290,-1.531541,-160.16879,-71.23536
4,0.304306,0.693806,0.000356,38.777512,29.912720,319.709665,1.456157,7.960031,2.100458,0.662759,0.682666,33.845108,-9.906712,-1.513181,-156.30979,-71.23536


De volgende 2 stukjes code moeten true terug geven (dit is ter controle) wel belangrijk!!!!

In [17]:
(df_biooracle['lat_deg'] == df_subset['lat_deg']).all()  # True als alle rijen exact overeenkomen

True

In [18]:
(df_biooracle['long_deg'] == df_subset['long_deg']).all() # True als alle rijden exact overeenkomen

True

De volgende code is afhankelijk van welke variabelen je in je model steekt (dus niet exact kopiëren)

In [19]:
df_subset = df_subset.rename(columns={
    "so_mean": "salinity_mean",
    "thetao_mean": "T_mean",
    "siconc_mean": "si_mean",
})
df_subset.columns

Index(['chl_mean', 'clt_mean', 'dfe_mean', 'mlotst_mean', 'no3_mean',
       'o2_mean', 'phyc_mean', 'ph_mean', 'po4_mean', 'si_mean',
       'sithick_mean', 'salinity_mean', 'tas_mean', 'T_mean', 'long_deg',
       'lat_deg'],
      dtype='object')

In [20]:
df_subset.head()

,chl_mean,clt_mean,dfe_mean,mlotst_mean,no3_mean,o2_mean,phyc_mean,ph_mean,po4_mean,si_mean,sithick_mean,salinity_mean,tas_mean,T_mean,long_deg,lat_deg
0,0.264738,0.685382,0.000333,59.873447,30.339450,317.311825,1.462501,7.956235,2.127915,0.727800,0.707284,33.942649,-11.655294,-1.580884,-171.74579,-71.23536
1,0.279545,0.698337,0.000307,59.631997,30.320359,319.230605,1.453052,7.957836,2.126779,0.720161,0.725998,33.953235,-11.305795,-1.583080,-167.88679,-71.23536
2,0.272520,0.703587,0.000304,52.292932,30.218967,318.271827,1.395905,7.957967,2.119736,0.681336,0.676522,33.936083,-10.861727,-1.538709,-164.02779,-71.23536
3,0.297756,0.698612,0.000315,31.966103,29.987898,320.985091,1.454854,7.960676,2.105292,0.677055,0.682275,33.892117,-10.390290,-1.531541,-160.16879,-71.23536
4,0.304306,0.693806,0.000356,38.777512,29.912720,319.709665,1.456157,7.960031,2.100458,0.662759,0.682666,33.845108,-9.906712,-1.513181,-156.30979,-71.23536


Nog de variabelen van env toevoegen die je ook nodig hebt voor in je model 

In [21]:
df_merged = df_subset.merge(
    df_env[['long_deg', 'lat_deg', 'DepthMean', 'LandDist', 'Slope', 'Abyssal']],
    on=['long_deg', 'lat_deg'],
    how='left'
)
df_merged.columns

Index(['chl_mean', 'clt_mean', 'dfe_mean', 'mlotst_mean', 'no3_mean',
       'o2_mean', 'phyc_mean', 'ph_mean', 'po4_mean', 'si_mean',
       'sithick_mean', 'salinity_mean', 'tas_mean', 'T_mean', 'long_deg',
       'lat_deg', 'DepthMean', 'LandDist', 'Slope', 'Abyssal'],
      dtype='object')

In [22]:
df_merged_vol = df_merged.merge(
    df_biooracle[['par_mean', 'kdpar_mean', 'terrain_characteristics_bea_mean', 'long_deg', 'lat_deg']],
    on=['long_deg', 'lat_deg'],
    how='left'
)
df_merged_vol.columns

Index(['chl_mean', 'clt_mean', 'dfe_mean', 'mlotst_mean', 'no3_mean',
       'o2_mean', 'phyc_mean', 'ph_mean', 'po4_mean', 'si_mean',
       'sithick_mean', 'salinity_mean', 'tas_mean', 'T_mean', 'long_deg',
       'lat_deg', 'DepthMean', 'LandDist', 'Slope', 'Abyssal', 'par_mean',
       'kdpar_mean', 'terrain_characteristics_bea_mean'],
      dtype='object')

Sla je dataset op als csv bestand dan kan je dit gebruiken in je ander documentje met je model 

In [23]:
df_merged_vol.to_csv("GD_2050-60_SSP126.csv")